# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', 'Dataset')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

A [RecordSet](https://mlcommons.org/croissant/1.0/recordset.html) points to a table of structured data (e.g. a CSV file or a table in a database).

We inspect all RecordSets, their `@id`, and available Fields (columns).

In [ ]:
# Inspect available record sets
record_sets = list(dataset.list_record_sets())
print('Available RecordSets:')
for rset in record_sets:
    print(f"  @id: {rset['@id']} | name: {rset.get('name', '<no name>')}")
    if 'field' in rset:
        print('    Fields:')
        for field in rset['field']:
            field_id = field['@id'] if isinstance(field, dict) else field
            # Try to retrieve extra field info
            field_info = dataset.get_entity(field_id)
            field_name = field_info.get('name', '<no name>') if isinstance(field_info, dict) else '<no name>'
            print(f"      {field_id} | name: {field_name}")
    else:
        print('    No fields found.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset, we extract all available record sets. Below, we demonstrate how to dynamically load records for each one.


In [ ]:
# Extract data from each record set
dataframes = {}
for rset in record_sets:
    rset_id = rset['@id']
    print(f"Loading RecordSet: {rset_id}")
    try:
        records_gen = dataset.records(record_set=rset_id)
        records = list(records_gen)
        if len(records) == 0:
            print('  No records found in this record set.')
            continue
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load data for {rset_id}: {e}")
        continue

# Show columns for the first available record set with data
if dataframes:
    first_rset_id = list(dataframes.keys())[0]
    print(f'Columns in {first_rset_id}:')
    print(dataframes[first_rset_id].columns.tolist())
    dataframes[first_rset_id].head()
else:
    print('No dataframes loaded. Check dataset source and available record sets.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Below we:
- Select a numeric field for analysis (e.g., a column representing abundance, MW, or similar)
- Filter records with that field above a threshold
- Normalize the numeric field
- (If available) Group by a categorical field (e.g., 'accession' or 'description')

__Replace below with actual field `@id`s from your data overview if needed.__

In [ ]:
# Example: perform EDA on the first record set with data
import numpy as np

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    # Attempt to select a useful numeric field dynamically
    # Common fields to test for biological/proteomics tables
    preferred_numeric_fields = ['MW', 'molecular_weight', 'abundance', 'abundance_raw', 'Abundance', 'Abundance_Sample1', 'Abundance_Sample2', 'unique_peptides', 'coverage_percent', 'PeptideCount', 'Score']
    available_numeric = [c for c in preferred_numeric_fields if c in df.columns]
    if not available_numeric:
        # Otherwise, pick the first numeric column
        candidate_numeric = df.select_dtypes(include=[np.number]).columns
        if len(candidate_numeric) == 0:
            print('No numeric fields available for EDA analysis.')
        else:
            numeric_field = candidate_numeric[0]
    else:
        numeric_field = available_numeric[0]
    
    # Now perform filtering/normalization
    if 'numeric_field' in locals():
        print(f'Using numeric field: {numeric_field}')
        if df[numeric_field].dtype not in [np.float64, np.int64, float, int]:
            # Try to coerce to numeric type
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()  # Example: use mean as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f'Filtered records with {numeric_field} > {threshold:.2f}:')
        print(filtered_df.head(3))

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

        # Pick a grouping field, e.g., 'accession' or 'description'
        preferred_group_fields = ['accession', 'description', 'group', 'sample', 'condition']
        group_field = None
        for pf in preferred_group_fields:
            if pf in filtered_df.columns:
                group_field = pf
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head(3))
        else:
            print('No suitable categorical field found for grouping.')
    else:
        print('No numeric field could be found for analysis.')
else:
    print('No available DataFrames for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and 'numeric_field' in locals():
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # If two or more numeric columns, show scatterplot
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 1:
        plt.figure(figsize=(6, 5))
        sns.scatterplot(data=df, x=num_cols[0], y=num_cols[1], alpha=0.7)
        plt.xlabel(num_cols[0])
        plt.ylabel(num_cols[1])
        plt.title(f"Scatter plot of {num_cols[0]} vs {num_cols[1]}")
        plt.show()
else:
    print('Not enough numeric data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load, inspect, and process the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. You:
- Inspected the schema and record sets using their `@id`s
- Loaded data into pandas DataFrames
- Explored and filtered numeric fields
- Visualized main distributions

For further analysis, you can select additional record sets or fields by their `@id`, or conduct more advanced feature engineering and machine learning workflows on the extracted tables.
